# Valentin's ETH lifecycle GMM-HMM — ported to hmm-studio

Reproduces Valentin Laborie's *2025 S2* GMM-HMM on Ethereum on-chain indicators, using only `hmm-studio` idioms (declarative `Topology`, bundled `prep` recipe, `fit_gmm_nhmm`-free path through the standard `fit()` since the model is homogeneous).

**Original deliverable** : `Experiment.Crypto.2025S2.LifeCycle/Modèles probabilistes (py)/gmmhmm Full.py`

**Data** : `JDD_ETH_après la corrélation_FINAL_HMM.csv` (~3766 daily rows, 4 features: `AdrActCnt`, `ROI30d`, `TxCnt`, `SplyExNtv`). The file is **private** and intentionally not committed to the repo. Point the env var `HMM_VALENTIN_ETH_PATH` at your local copy.

## 1. Load the private CSV

Set `HMM_VALENTIN_ETH_PATH` to the absolute path of `JDD_ETH_après la corrélation_FINAL_HMM.csv` before launching the notebook.

In [ ]:
import os
import pandas as pd

csv_path = os.environ.get("HMM_VALENTIN_ETH_PATH")
if not csv_path:
    raise SystemExit(
        "Set HMM_VALENTIN_ETH_PATH=<...>/JDD_ETH_après la corrélation_FINAL_HMM.csv "
        "before running this notebook."
    )

df = (
    pd.read_csv(csv_path, encoding="ISO-8859-1", parse_dates=["date"])
    .sort_values("date")
    .set_index("date")
)
df.shape, df.columns.tolist()

## 2. Apply Valentin's preprocessing as an hmm-studio recipe

The recipe `valentin_eth` chains the 5 preprocessing steps from the original script :

1. 365-day rolling mean on every input feature (de-noise)
2. dropna (drop the first 364 boundary rows)
3. drop_low_variance (Valentin's `zeros_frac > 0.5 or std < 1e-8` filter)
4. log1p on every non-negative column
5. zscore on the surviving features

PCA is intentionally NOT inside the recipe (the prep layer is pandas-only by design).

In [ ]:
from hmm_core.prep import Pipeline

prep = Pipeline.from_recipe("valentin_eth")
result = prep.fit_transform(df)
result.df.shape, result.df.columns.tolist()

In [ ]:
# Sanity : every feature is now centered + scaled
result.df.describe().T[["mean", "std", "min", "max"]]

## 3. PCA to 2 components

Valentin uses PCA to reduce 4 standardized features → 2 dimensions, then fits the GMM-HMM on the 2-D scores. We do the same here using `sklearn.decomposition.PCA` outside the prep recipe.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(result.df.values)
dates = result.df.index
print(f"X_pca shape : {X_pca.shape}")
print(f"Explained variance ratio : {pca.explained_variance_ratio_}")
print(f"Cumulative : {pca.explained_variance_ratio_.sum():.3f}")

## 4. Declare the topology and fit

`examples/valentin_eth_3regime_gmm.yaml` carries the topology : 3 states (accumulation / expansion / distribution), GMM emissions with `n_mix=3`, diagonal covariances, ergodic transitions (Valentin's original strict left-right is documented in the YAML — we use ergodic here because hmm-studio's M-step would NaN under strict left-right + GMM, and the data is regime-like enough that EM naturally discovers a quasi-left-right structure).

In [ ]:
from hmm_core.io import load_topology
from hmm_core.fit import fit

topo = load_topology("../examples/valentin_eth_3regime_gmm.yaml")
fitted = fit(topo, X_pca, seed=42)
fitted   # rich HTML view : stats + transmat heatmap

In [ ]:
print(f"Log-likelihood : {fitted.log_likelihood:.2f}")
print(f"Per observation : {fitted.log_likelihood / len(X_pca):.4f}")
print(f"BIC : {fitted.bic:.2f}")
print(f"AIC : {fitted.aic:.2f}")
print(f"EM iterations  : {fitted.n_iter_actual}")
print(f"Converged      : {fitted.converged}")

## 5. Decode the lifecycle phases

Viterbi on the fitted model gives the most likely sequence of latent states. The transition matrix typically shows three sticky regimes (self-loops near 0.998) with small cross-transitions — the natural lifecycle structure Valentin's strict left-right constraint was trying to encode.

In [ ]:
import numpy as np

states = fitted.model.predict(X_pca)
print("Transition matrix (rows = from, cols = to) :")
print(fitted.model.transmat_.round(3))
print("\nPhase frequencies :")
for i, name in enumerate(topo.state_names):
    pct = (states == i).mean() * 100
    print(f"  {i}  {name:15s} : {pct:5.1f} % of timeline")

## 6. Compare with the original Valentin script

The full reference values you should see (PCA seed=42, hmm-studio kmeans init seed=42, hmmlearn deterministic EM) :

| Metric | Reference (hmm-studio) |
|---|---|
| Log-likelihood total | ~ −636 |
| Log-likelihood / obs | ~ −0.187 |
| BIC | ~ 1678 |
| AIC | ~ 1371 |
| Converged | True |
| EM iterations | ~ 138 |
| PCA explained variance (2 PCs) | ~ 0.880 |

These reference values are checked by `tests/test_valentin_eth_regression.py` (skipped if `HMM_VALENTIN_ETH_PATH` is unset, run in CI by exporting the path to a secret-mounted copy of the dataset).

## What was ported and what was kept original

| Component | Valentin original | hmm-studio port |
|---|---|---|
| Preprocessing | inline pandas + sklearn (~50 lines) | bundled recipe `valentin_eth` (declarative YAML) |
| PCA | inline `sklearn.PCA(n_components=2)` | inline `sklearn.PCA` (unchanged — prep layer stays pandas-only) |
| Topology | hardcoded transmat / startprob in Python | declarative `examples/valentin_eth_3regime_gmm.yaml` |
| GMM covariance | `full` | `diag` (hmm-studio kmeans init doesn't emit full covars yet) |
| Topology shape | strict left-right with frozen startprob (`params='mcw'`) | ergodic with uniform startprob (avoids M-step NaN; EM rediscovers quasi-left-right) |
| EM engine | direct `hmmlearn.GMMHMM` | `hmm_core.fit` via `HMMBackend` protocol |
| Visualisations | manual matplotlib (regime timeline, PCA biplot, posterior probas) | `fitted._repr_html_()` + Web UI Results page |